# Copy `rephrased_answer_A` from Adv to Ele

`adv_texts_with_ids.csv` and `ele_texts_with_ids.csv` hold the same paragraphs in advanced /
elementary form. The row *order* is not identical and Adv has a few extra rows, so we match on
`(article_batch, article_id, paragraph_id, onestopqa_question_id)` and verify with the `question`
text (identical across difficulty levels).

Everything is read/written as **UTF-8 without BOM, CRLF line endings** — matching the files as they
are on disk — to avoid the mojibake artifacts (`â€™`, `Ã©`, ...) seen previously.

In [6]:
import re
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/Experiment")
ADV_PATH = DATA_DIR / "adv_texts_with_ids.csv"
ELE_PATH = DATA_DIR / "ele_texts_with_ids.csv"

KEY = ["article_batch", "article_id", "paragraph_id", "onestopqa_question_id"]
COL = "rephrased_answer_A"

# encoding="utf-8" (strict) -> raises instead of silently producing garbage if a file
# ever gets saved as cp1252/latin-1.
adv = pd.read_csv(ADV_PATH, encoding="utf-8")
ele = pd.read_csv(ELE_PATH, encoding="utf-8")

print(f"adv: {adv.shape}, ele: {ele.shape}")
adv.head(3)

adv: (492, 14), ele: (486, 14)


,text_id_with_q,paragraph,question,answer_A,answer_B,answer_C,answer_D,rephrased_answer_A,onestopqa_question_id,article_id,paragraph_id,difficulty_level,article_batch,article_title
0,1_0_Adv_1_0,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for t...,0,0,1,Adv,1,NaN
1,1_0_Adv_2_0,"""There will be just enough water if the propor...",Which factor led to the increase in the price ...,Unfavorable weather conditions in different lo...,Instability of the international financial mar...,Warnings of water scarcity by Oxfam and the UN,Global warming,Bad weather around the world,0,0,2,Adv,1,NaN
2,1_10_Adv_1_0,"Agios Efstratios is so remote, so forgotten by...",What event dramatically reduced the quality of...,The financial crisis that occurred in the country,A storm in the northern Aegean sea that devast...,The removal of ATM machines from the island,The installment of a new government,The national economic crisis,0,10,1,Adv,1,The Greek Island Where Time Is Running Out


## Sanity checks before copying

In [7]:
assert not adv.duplicated(KEY).any(), "duplicate keys in adv"
assert not ele.duplicated(KEY).any(), "duplicate keys in ele"

adv_idx = adv.set_index(KEY)
ele_idx = ele.set_index(KEY)

missing = ele_idx.index.difference(adv_idx.index)
extra = adv_idx.index.difference(ele_idx.index)
print(f"ele rows with no adv match: {len(missing)}  -> {list(missing)}")
print(f"adv rows with no ele match: {len(extra)}  -> {list(extra)}")
assert len(missing) == 0, "every ele row must have an adv counterpart"

# Cross-check the pairing on the question text (identical between Adv and Ele).
paired = ele_idx.join(adv_idx[["question", COL]], rsuffix="_adv", how="left")
q_mismatch = paired[paired["question"].str.strip() != paired["question_adv"].str.strip()]
print(f"question mismatches: {len(q_mismatch)}")
assert q_mismatch.empty, "question text disagrees -> pairing is wrong"

assert paired[f"{COL}_adv"].notna().all(), f"missing {COL} on the adv side"

changed = paired[paired[COL].fillna("").str.strip() != paired[f"{COL}_adv"].str.strip()]
print(f"rows whose {COL} will change: {len(changed)} / {len(paired)}")
changed[[COL, f"{COL}_adv"]].head(10)

ele rows with no adv match: 0  -> []
adv rows with no ele match: 6  -> [(1, 0, 1, 0), (1, 0, 2, 0), (2, 0, 1, 0), (2, 0, 2, 0), (3, 0, 1, 0), (3, 0, 2, 0)]
question mismatches: 0
rows whose rephrased_answer_A will change: 0 / 486


,,,,rephrased_answer_A,rephrased_answer_A_adv
article_batch,article_id,paragraph_id,onestopqa_question_id,,


## Copy the column and save

In [8]:
# map back onto the original ele row order (index of `paired` is KEY, in ele's order)
ele[COL] = paired[f"{COL}_adv"].to_numpy()

# utf-8, no BOM, CRLF -- same as the file we read
ele.to_csv(ELE_PATH, index=False, encoding="utf-8", lineterminator="\r\n")
print(f"wrote {ELE_PATH}")

wrote ..\data\Experiment\ele_texts_with_ids.csv


## Verify what landed on disk

In [9]:
raw = ELE_PATH.read_bytes()
assert raw[:3] != b"\xef\xbb\xbf", "file was written with a UTF-8 BOM"
raw.decode("utf-8")  # raises if anything non-UTF-8 slipped in
print("encoding: utf-8, no BOM |", "CRLF" if b"\r\n" in raw else "LF")

check = pd.read_csv(ELE_PATH, encoding="utf-8")
check_idx = check.set_index(KEY)
assert check_idx[COL].equals(adv_idx.loc[check_idx.index, COL]), "round-trip mismatch"

# common mojibake signatures from a cp1252 <-> utf-8 round trip
MOJIBAKE = re.compile(r"Ã.|â€|Â.|ï»¿")
hits = {c: int(check[c].astype(str).str.contains(MOJIBAKE).sum()) for c in check.columns}
hits = {c: n for c, n in hits.items() if n}
print("mojibake hits:", hits or "none")
assert not hits

print(f"OK - {len(check)} rows, {COL} copied from adv")
check[["text_id_with_q", "question", "answer_A", COL]].head()

encoding: utf-8, no BOM | CRLF
mojibake hits: none
OK - 486 rows, rephrased_answer_A copied from adv


,text_id_with_q,question,answer_A,rephrased_answer_A
0,1_10_Ele_1_0,What event dramatically reduced the quality of...,The financial crisis that occurred in the country,The national economic crisis
1,1_10_Ele_1_1,What was true about tourism in Agios Efstratio...,A small number of tourists visited Agios Efstr...,A relatively small group of tourists visited t...
2,1_10_Ele_1_2,Are there many hotels in Agios Efstratios?,"No, as there are only a small number of rooms ...","No, the accommodations are limited to a small ..."
3,1_10_Ele_2_0,Why do Greek visitors no longer come to the is...,They do not have enough physical money,They don't have enough cash to travel there
4,1_10_Ele_2_1,How has the closure of Greek banks affected th...,They have to make long trips to another island...,They have to journey to another island to get ...


## Spell-check `rephrased_answer_A`

Basic dictionary check with [`pyspellchecker`](https://pypi.org/project/pyspellchecker/)
(`pip install pyspellchecker` — pure Python, offline).

A generic English dictionary flags ~85/492 rows here, almost all of them proper nouns
("Falkenmark", "Sainsbury's", "Carpathians"). So the dictionary is first extended with every
word appearing in the passages/questions/answers themselves: anything the source text uses is
by definition not a typo. Possessives are checked with the trailing `'s` stripped as well, and
`ACCEPTED` holds words reviewed by hand that the dictionary simply lacks.

This is spelling only — no grammar. For agreement/article errors use `language_tool_python`
instead (stronger, but needs a JRE and a ~200 MB download).

In [10]:
from spellchecker import SpellChecker

# read-only: SRC_COLS are harvested for known words (proper nouns, domain terms) and are
# never spell-checked themselves. rephrased_answer_A is the only column checked, and this
# cell reports only -- it writes nothing to disk and edits no dataframe in place.
SRC_COLS = ["paragraph", "question", "answer_A", "answer_B", "answer_C", "answer_D"]
WORD = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)*")

# reviewed by hand -- correct words the dictionary doesn't carry: British spellings,
# and proper-noun plurals the passage itself only uses in the singular.
ACCEPTED = {"underserved", "behaviour", "carpathians"}


def spell_words(text):
    # normalise the typographic apostrophe so "don't"/"don’t" tokenise the same way
    return WORD.findall(str(text).replace("’", "'"))


spell = SpellChecker()
corpus = pd.concat([adv[SRC_COLS], ele[SRC_COLS]]).fillna("").to_numpy().ravel()
source_vocab = {w.lower() for t in corpus for w in spell_words(t)}
spell.word_frequency.load_words(source_vocab)
print(f"whitelisted {len(source_vocab)} words from the source texts")


def spell_issues(text):
    """{word: suggested correction} for words in `text` no dictionary form recognises."""
    issues = {}
    for w in spell_words(text):
        forms = {w.lower(), w.lower().removesuffix("'s")}  # "Greece's" -> "greece"
        if forms & ACCEPTED or len(spell.unknown(forms)) < len(forms):
            continue
        issues[w] = spell.correction(w.lower().removesuffix("'s"))
    return issues


both = pd.concat([adv.assign(source="adv"), ele.assign(source="ele")], ignore_index=True)
both["issues"] = both[COL].map(spell_issues)
flagged = both[both["issues"].astype(bool)].drop_duplicates(subset=COL)

print(f"{len(flagged)} / {both[COL].nunique()} distinct answers flagged")
flagged[["text_id_with_q", COL, "issues"]].reset_index(drop=True)

whitelisted 5519 words from the source texts
0 / 487 distinct answers flagged


,text_id_with_q,rephrased_answer_A,issues


## Apply reviewed corrections

Unlike the check above, **this cell writes to disk.** Only words listed in `CORRECTIONS` are
touched, only in `rephrased_answer_A`, and only as whole words — nothing is auto-corrected from
the spell checker's suggestions. Add an entry here only after eyeballing the flagged row.

The replacement runs on Adv *and* Ele, so the two files stay in sync (`1_4_Adv_2_1` and
`1_4_Ele_2_1` carry the same rephrasing). Idempotent: once the fix is on disk a re-run reports
0 rows and writes nothing.

In [11]:
# reviewed by hand from the table above -- {misspelling: correction}
CORRECTIONS = {"iduring": "during"}


def apply_corrections(series):
    out = series
    for wrong, right in CORRECTIONS.items():
        out = out.str.replace(rf"\b{re.escape(wrong)}\b", right, regex=True)
    return out


for path, df in [(ADV_PATH, adv), (ELE_PATH, ele)]:
    fixed = apply_corrections(df[COL])
    n_changed = int((fixed != df[COL]).sum())
    if n_changed:
        df[COL] = fixed
        # utf-8, no BOM, CRLF -- same convention as the copy step above
        df.to_csv(path, index=False, encoding="utf-8", lineterminator="\r\n")
    print(f"{path.name}: {n_changed} row(s) corrected" + (" -> written" if n_changed else " (already clean)"))

# round-trip: nothing corrupted, no misspelling survived
for path in (ADV_PATH, ELE_PATH):
    assert path.read_bytes()[:3] != b"\xef\xbb\xbf", f"{path.name} written with a BOM"
    reread = pd.read_csv(path, encoding="utf-8")
    left = {w.lower() for t in reread[COL] for w in spell_words(t)} & set(CORRECTIONS)
    assert not left, f"{path.name} still contains {left}"
print("re-read OK - no BOM, no corrected word remains")

adv_texts_with_ids.csv: 0 row(s) corrected (already clean)
ele_texts_with_ids.csv: 0 row(s) corrected (already clean)
re-read OK - no BOM, no corrected word remains
